# SR-01 — Predicted-state feedback preflight 14

Tre GRU standard con parametri e inizializzazione identici. Cambia soltanto il contratto informativo: nessuno stato, stato vero solo al primo passo, oppure feedback esclusivamente della propria previsione. Training GPU riprendibile, ETA per epoca, validation only e test chiuso. Serve soltanto `hay_micro_4c_event_enriched_v2.h5`.

In [ ]:
from pathlib import Path
import subprocess, sys
URL='https://github.com/Zagred47/LearningSingleCompartiment.git'
ROOT=Path('/kaggle/working/LearningSingleCompartiment_sr01')
if (ROOT/'pyproject.toml').is_file():
    subprocess.run(['git','-C',str(ROOT),'pull','--ff-only','origin','main'],check=True)
else:
    if ROOT.exists() and any(ROOT.iterdir()):
        raise RuntimeError(f'{ROOT} esiste ma non è una clone valida; scegli un runtime pulito o rimuovila manualmente')
    subprocess.run(['git','clone',URL,str(ROOT)],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','--no-build-isolation','-e',str(ROOT)],check=True)
REPO_ROOT=ROOT
print('Repository:',REPO_ROOT)

In [ ]:
import os
datasets=list(Path('/kaggle/input').glob('**/*.h5'))
print('Dataset trovati:'); [print(' ',p) for p in datasets]
os.environ['HAY_SR01_OUTPUT']='/kaggle/working/hay_micro_state_feedback_preflight_14'
# Opzionale per un test rapido, non per il risultato ufficiale:
# os.environ['HAY_SR01_EPOCHS']='2'
# os.environ['HAY_SR01_MAX_TRAIN_TRAJECTORIES']='6'
# os.environ['HAY_SR01_MAX_VALIDATION_TRAJECTORIES']='4'
# Per includere i tre checkpoint migliori nello ZIP finale (circa 20 MiB):
# os.environ['HAY_SR01_DOWNLOAD_CHECKPOINTS']='1'

In [ ]:
import runpy
result=runpy.run_path(str(REPO_ROOT/'notebooks/micro_state_feedback_preflight_14.py'))
OUTPUT_DIR=Path(result['OUTPUT']); ZIP_PATH=Path(result['ZIP_PATH'])
print('Output:',OUTPUT_DIR); print('ZIP:',ZIP_PATH)

In [ ]:
import json, pandas as pd
from IPython.display import Image,display
display(pd.read_csv(OUTPUT_DIR/'validation_comparison.csv'))
display(pd.Series(json.loads((OUTPUT_DIR/'decision.json').read_text())))
for name in ('learning_curves.png','event_trace.png'):
    path=OUTPUT_DIR/'figures'/name
    if path.exists(): display(Image(filename=str(path)))

In [ ]:
from IPython.display import FileLink,Javascript,display
import base64
print('Archivio:',ZIP_PATH, f'({ZIP_PATH.stat().st_size/2**20:.1f} MiB)')
display(FileLink(str(ZIP_PATH)))
encoded=base64.b64encode(ZIP_PATH.read_bytes()).decode('ascii'); filename=ZIP_PATH.name
display(Javascript(f"""
const binary=atob('{encoded}'); const bytes=new Uint8Array(binary.length);
for(let i=0;i<binary.length;i++) bytes[i]=binary.charCodeAt(i);
const blob=new Blob([bytes],{{type:'application/zip'}}); const url=URL.createObjectURL(blob);
const a=document.createElement('a'); a.href=url; a.download='{filename}'; document.body.appendChild(a); a.click(); a.remove();
setTimeout(()=>URL.revokeObjectURL(url),60000);
"""))